In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from sklearn.cluster import KMeans
from sklearn.linear_model import Ridge
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import pygeohash as pgh
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("Loading data...")
train = pd.read_csv('dataset/train.csv')
test  = pd.read_csv('dataset/test.csv')

# ─────────────────────────────────────────────
# 2. ADVANCED FEATURES & UNSUPERVISED CLUSTERING
# ─────────────────────────────────────────────
def build_base_features(df):
    df = df.copy()

    # --- Spatial ---
    df['lat'] = df['geohash'].apply(lambda x: pgh.decode(x)[0] if pd.notnull(x) else np.nan)
    df['lon'] = df['geohash'].apply(lambda x: pgh.decode(x)[1] if pd.notnull(x) else np.nan)
    df['geo_region'] = df['geohash'].str[:4]
    df['geo_subregion'] = df['geohash'].str[:5]
    
    # --- Time & Weekly Cycles ---
    ts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour']        = ts[0]
    df['time_slot']   = (ts[0] * 60 + ts[1]) // 15 
    
    df['time_sin'] = np.sin(2 * np.pi * df['time_slot'] / 96)
    df['time_cos'] = np.cos(2 * np.pi * df['time_slot'] / 96)
    df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24) 
    df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24) 
    
    df['day_of_week'] = df['day'] % 7
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
    
    # Peak hour flags
    df['is_morning_peak'] = ((df['hour'] >= 7)  & (df['hour'] <= 9)).astype(int)
    df['is_evening_peak'] = ((df['hour'] >= 17) & (df['hour'] <= 19)).astype(int)

    # ★ THE HOLY GRAIL KEYS ★
    df['geo_time_key']     = df['geohash'] + "_" + df['time_slot'].astype(str)
    # Location + Day of Week + Exact Time (The ultimate traffic footprint)
    df['geo_dow_time_key'] = df['geohash'] + "_" + df['day_of_week'].astype(str) + "_" + df['time_slot'].astype(str)

    # --- Pure Categoricals (For Native Processing) ---
    df['LargeVehicles'] = df['LargeVehicles'].fillna('Unknown').astype('category')
    df['Landmarks']     = df['Landmarks'].fillna('Unknown').astype('category')
    df['RoadType']      = df['RoadType'].fillna('Unknown').astype('category')
    df['Weather']       = df['Weather'].fillna('Unknown').astype('category')

    # --- Temperature ---
    df['Temperature']  = df.groupby([df['geohash'].str[:4], 'day'])['Temperature'].transform(lambda x: x.fillna(x.median()))
    df['Temperature']  = df['Temperature'].fillna(df['Temperature'].median())
    df['temp_x_lanes'] = df['Temperature'] * df['NumberofLanes']

    return df

print("Building base features...")
train_base = build_base_features(train)
test_base  = build_base_features(test)

print("Generating spatial clusters...")
kmeans = KMeans(n_clusters=60, random_state=42, n_init=10)
train_base['spatial_cluster'] = kmeans.fit_predict(train_base[['lat', 'lon']].fillna(0))
test_base['spatial_cluster']  = kmeans.predict(test_base[['lat', 'lon']].fillna(0))

# ─────────────────────────────────────────────
# 3. STRICT OOF TARGET ENCODING
# ─────────────────────────────────────────────
def get_oof_target_encoding(train_df, test_df, columns_to_encode, target_col):
    tr_encoded = train_df.copy()
    te_encoded = test_df.copy()
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    global_mean = train_df[target_col].mean()

    for col in columns_to_encode:
        new_col_mean = f'{col}_target_mean'
        tr_encoded[new_col_mean] = np.nan
        
        for tr_idx, val_idx in kf.split(train_df):
            X_tr = train_df.iloc[tr_idx]
            fold_means = X_tr.groupby(col)[target_col].mean().to_dict()
            tr_encoded.loc[val_idx, new_col_mean] = train_df.loc[val_idx, col].map(fold_means)
            
        tr_encoded[new_col_mean] = tr_encoded[new_col_mean].fillna(global_mean)
        full_means = train_df.groupby(col)[target_col].mean().to_dict()
        te_encoded[new_col_mean] = test_df[col].map(full_means).fillna(global_mean)

    return tr_encoded, te_encoded

print("Applying Out-Of-Fold Target Encoding...")
# Added the Weekly Cycle Key here
encode_cols = ['geohash', 'spatial_cluster', 'geo_time_key', 'geo_dow_time_key']
train_encoded, test_encoded = get_oof_target_encoding(train_base, test_base, encode_cols, 'demand')

# ─────────────────────────────────────────────
# 4. TRIPLE-MODEL LEVEL-1 TRAINING (RAW TARGET)
# ─────────────────────────────────────────────
FEATURES = [
    'lat', 'lon', 'spatial_cluster', 
    'time_slot', 'day', 'day_of_week', 'is_weekend', 'hour', 
    'hour_sin', 'hour_cos', 'time_sin', 'time_cos',
    'is_morning_peak', 'is_evening_peak', 
    'LargeVehicles', 'Landmarks', 'RoadType', 'Weather', # Passed as raw categories
    'NumberofLanes', 'Temperature', 'temp_x_lanes',
    'geohash_target_mean', 'spatial_cluster_target_mean', 
    'geo_time_key_target_mean', 'geo_dow_time_key_target_mean' # New ultra-granular feature
]

CATEGORICAL_COLS = ['LargeVehicles', 'Landmarks', 'RoadType', 'Weather']

X = train_encoded[FEATURES]
y = train_encoded['demand'] # NO LOG TRANSFORM. Attack the peaks.
X_test = test_encoded[FEATURES]

kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_lgb, test_preds_lgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_xgb, test_preds_xgb = np.zeros(len(X)), np.zeros(len(X_test))
oof_cat, test_preds_cat = np.zeros(len(X)), np.zeros(len(X_test))

print("\n── Training Level 1 Models ──")
for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # --- LightGBM ---
    model_lgb = lgb.LGBMRegressor(
        objective='regression', metric='rmse', n_estimators=2500, learning_rate=0.025, 
        num_leaves=255, min_child_samples=20, feature_fraction=0.8, bagging_fraction=0.8, 
        bagging_freq=5, reg_alpha=0.2, reg_lambda=0.2, verbose=-1, n_jobs=-1
    )
    # LGBM handles pandas categories automatically
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(150, verbose=False)])
    oof_lgb[val_idx] = model_lgb.predict(X_val)
    test_preds_lgb += model_lgb.predict(X_test) / 5

    # --- XGBoost ---
    # Convert categories to native XGBoost 'category' type
    X_tr_xgb = X_tr.copy()
    X_val_xgb = X_val.copy()
    X_test_xgb = X_test.copy()
    for col in CATEGORICAL_COLS:
        X_tr_xgb[col] = X_tr_xgb[col].cat.codes
        X_val_xgb[col] = X_val_xgb[col].cat.codes
        X_test_xgb[col] = X_test_xgb[col].cat.codes

    model_xgb = xgb.XGBRegressor(
        objective='reg:squarederror', eval_metric='rmse', n_estimators=2500, learning_rate=0.025, 
        max_depth=8, min_child_weight=10, colsample_bytree=0.8, subsample=0.8, 
        reg_alpha=0.2, reg_lambda=0.2, n_jobs=-1, random_state=42
    )
    model_xgb.fit(X_tr_xgb, y_tr, eval_set=[(X_val_xgb, y_val)], verbose=False)
    oof_xgb[val_idx] = model_xgb.predict(X_val_xgb)
    test_preds_xgb += model_xgb.predict(X_test_xgb) / 5

    # --- CatBoost ---
    # Pass column names to CatBoost natively
    model_cat = CatBoostRegressor(
        iterations=2500, learning_rate=0.03, depth=7, l2_leaf_reg=3,
        cat_features=CATEGORICAL_COLS, eval_metric='RMSE', random_seed=42, verbose=False, thread_count=-1
    )
    model_cat.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], early_stopping_rounds=150)
    oof_cat[val_idx] = model_cat.predict(X_val)
    test_preds_cat += model_cat.predict(X_test) / 5
    
    print(f"  Fold {fold+1} Base Models Completed")

# ─────────────────────────────────────────────
# 5. LEVEL-2 STACKING (THE META-LEARNER)
# ─────────────────────────────────────────────
print("\n── Training Level 2 Meta-Model (Ridge Stacking) ──")

X_level2 = pd.DataFrame({'lgb': oof_lgb, 'xgb': oof_xgb, 'cat': oof_cat})
X_test_level2 = pd.DataFrame({'lgb': test_preds_lgb, 'xgb': test_preds_xgb, 'cat': test_preds_cat})

meta_model = Ridge(alpha=1.0)
meta_model.fit(X_level2, y)

final_oof = meta_model.predict(X_level2)
final_test_preds = meta_model.predict(X_test_level2)

# Ensure no negative demand
final_oof = np.clip(final_oof, 0, None)
final_test_preds = np.clip(final_test_preds, 0, None)

final_oof_score = max(0, 100 * r2_score(y, final_oof))

print(f"\n★ Meta-Model Weights (LGB, XGB, CAT): {meta_model.coef_}")
print(f"★ Final Stacked OOF R²: {r2_score(y, final_oof):.6f}")
print(f"★ Expected Leaderboard Score: ~{final_oof_score:.4f}")

# ─────────────────────────────────────────────
# 6. SUBMISSION
# ─────────────────────────────────────────────
submission = pd.DataFrame({'Index': test['Index'], 'demand': final_test_preds})
submission.to_csv('submission_crown.csv', index=False)
print("Saved submission_crown.csv.")

Loading data...
Building base features...
Generating spatial clusters...
Applying Out-Of-Fold Target Encoding...

── Training Level 1 Models ──
  Fold 1 Base Models Completed
  Fold 2 Base Models Completed
  Fold 3 Base Models Completed
  Fold 4 Base Models Completed
  Fold 5 Base Models Completed

── Training Level 2 Meta-Model (Ridge Stacking) ──

★ Meta-Model Weights (LGB, XGB, CAT): [0.26934401 0.48918729 0.24719545]
★ Final Stacked OOF R²: 0.954074
★ Expected Leaderboard Score: ~95.4074
Saved submission_crown.csv.
